# Vocational NMT: run 7, third seed + LoRA rank sweep

Same data as run 6 (case-fixed formal drafts, shipped in the bundle); no translation of training data.

**GPU 0:** `joint_formal_v2_s44`, the main joint model with seed 44 → domain evaluation (plain and hybrid). With seeds 42 and 43 from run 6, this gives a 3-seed mean and spread.
**GPU 1:** Hindi-only LoRA rank sweep r = 4, 8, 32, 64 (alpha = 2r; r = 16 exists from run 6) → Hindi evaluation of each.

**Input:** only the latest bundle dataset. Use GPU T4 x2, Internet on, and the `HF_TOKEN` secret. Then **Save & Run All** (about 2.5 hours); the outputs end up in `/kaggle/working/outputs.zip`.

In [ ]:
# 1. Copy the newest stamped bundle into a writable working directory (+ reuse earlier drafts)
import os, shutil, subprocess, sys, glob, zipfile
from pathlib import Path

PROJECT = Path("/kaggle/working/project")
# Only folders/zips carrying kaggle/BUNDLE_VERSION count as code; outputs of earlier runs
# (added as inputs to reuse drafts) also contain a backend/ folder with OLD code.
candidates = []
for z in glob.glob("/kaggle/input/**/vocational-nmt-bundle.zip", recursive=True):
    with zipfile.ZipFile(z) as zf:
        if "kaggle/BUNDLE_VERSION" in zf.namelist():
            candidates.append((zf.read("kaggle/BUNDLE_VERSION").decode().strip(), z))
for stamp in glob.glob("/kaggle/input/**/kaggle/BUNDLE_VERSION", recursive=True):
    candidates.append((Path(stamp).read_text().strip(), str(Path(stamp).parents[1])))
if not candidates:
    raise FileNotFoundError("No bundle with kaggle/BUNDLE_VERSION under /kaggle/input - upload the latest bundle zip.")
print("bundles attached:", sorted({v for v, _ in candidates}))
version, source = max(candidates)

# /kaggle/working survives while an interactive session is open: re-copy when the attached
# bundle is newer than the copy from an earlier cell run.
copied = PROJECT / "kaggle/BUNDLE_VERSION"
if PROJECT.exists() and (not copied.exists() or copied.read_text().strip() != version):
    print("Replacing stale project copy", copied.read_text().strip() if copied.exists() else "(unstamped)")
    shutil.rmtree(PROJECT)
if not PROJECT.exists():
    print(f"Using bundle {version} from {source}")
    if source.endswith(".zip"):
        zipfile.ZipFile(source).extractall(PROJECT)
    else:
        shutil.copytree(source, PROJECT)
print("bundle version:", copied.read_text().strip())
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(PROJECT)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation during beam search
sys.path.insert(0, str(PROJECT))

CORPUS = PROJECT / "backend/data/corpus_v2"
for pattern, sub in (("drafts_*.csv", ""), ("postedit/test_*.csv", "postedit")):
    for src in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        dst = CORPUS / sub / Path(src).name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src, dst)
            print("reused", src)
print(sorted(p.name for p in CORPUS.iterdir()))
assert (PROJECT / "backend/models/it2_lora_joint/adapter_model.safetensors").exists(), "run-4 adapter missing from bundle"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install the pinned stack (about 1 minute)
!pip install -q -r kaggle/requirements-kaggle.txt
!python -c "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"

In [ ]:
# 3. Hugging Face token + access check for the gated repos
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import hf_hub_download
IN22_OK = True
for repo, kind, fname in [("ai4bharat/indictrans2-en-indic-dist-200M", "model", "config.json"),
                          ("ai4bharat/indictrans2-en-indic-1B", "model", "config.json"),
                          ("ai4bharat/IN22-Gen", "dataset", "data/train-00000-of-00001.parquet")]:
    try:
        hf_hub_download(repo, fname, repo_type=kind, token=os.environ["HF_TOKEN"])
        print("access OK:", repo)
    except Exception as exc:
        print(f"NO ACCESS: {repo} -> accept its terms on huggingface.co ({type(exc).__name__})")
        if kind == "model":
            raise
        IN22_OK = False

In [ ]:
# 4. Smoke test (~3 min): 10 training steps at the smallest rank, then load the adapter
def check(step):
    assert _exit_code == 0, f"{step} failed - see the output above"

for s in ["hi", "gom", "mai", "doi"]:
    assert (CORPUS / f"drafts_{s}_formal.csv").exists(), f"drafts_{s}_formal.csv missing from the bundle"
!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 4 --lora-alpha 8 --run-name smoke --max-steps 10 --eval-steps 5 --dev-per-lang 8
check("training smoke test")
!python -m backend.baseline_inference --model backend/models/it2_lora_smoke --text "Test the insulation with a megger." --target-lang hin_Deva
check("adapter loading")
shutil.rmtree("backend/models/it2_lora_smoke", ignore_errors=True)

## Training: GPU 0 third seed, GPU 1 rank sweep (each rank trained, then evaluated)

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
gpu1 = None
sweep = 'python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 4 --lora-alpha 8 --run-name hi_formal_v2_r4 && python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2_r4 --tag run7_r4 && python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 8 --lora-alpha 16 --run-name hi_formal_v2_r8 && python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2_r8 --tag run7_r8 && python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 32 --lora-alpha 64 --run-name hi_formal_v2_r32 && python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2_r32 --tag run7_r32 && python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 64 --lora-alpha 128 --run-name hi_formal_v2_r64 && python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2_r64 --tag run7_r64'
if n_gpu >= 2:
    gpu1 = subprocess.Popen(["bash", "-c", sweep], env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"),
                            stdout=open("/kaggle/working/rank_sweep_gpu1.log", "w"), stderr=subprocess.STDOUT)
else:
    print("Only one GPU: the rank sweep runs after the seed-44 model.")

!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --drafts-variant formal --run-name joint_formal_v2_s44 --seed 44
check("seed-44 fine-tuning")
!CUDA_VISIBLE_DEVICES=0 python -m backend.run_domain_eval --systems lora_joint_formal_v2_s44,lora_joint_formal_v2_s44+hy --tag run7_s44
check("seed-44 evaluation")

if gpu1 is not None:
    print("rank sweep exit code:", gpu1.wait())
    print(open("/kaggle/working/rank_sweep_gpu1.log").read()[-3000:])
else:
    !CUDA_VISIBLE_DEVICES=0 bash -c 'python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 4 --lora-alpha 8 --run-name hi_formal_v2_r4 && python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2_r4 --tag run7_r4 && python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 8 --lora-alpha 16 --run-name hi_formal_v2_r8 && python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2_r8 --tag run7_r8 && python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 32 --lora-alpha 64 --run-name hi_formal_v2_r32 && python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2_r32 --tag run7_r32 && python -m backend.finetune --drafts-variant formal --langs hin_Deva --lora-r 64 --lora-alpha 128 --run-name hi_formal_v2_r64 && python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2_r64 --tag run7_r64'

In [ ]:
# Package everything that must come back to your machine
out = Path("/kaggle/working/outputs")
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
for src in sorted(Path("backend/models").glob("it2_lora_*")):
    if src.name in {"it2_lora_joint"}:  # shipped in the bundle, not produced here
        continue
    shutil.copytree(src, out / src.name, ignore=shutil.ignore_patterns("checkpoints"))
shutil.copytree("backend/results", out / "results")
for log in glob.glob("/kaggle/working/*.log"):
    shutil.copy(log, out)
shutil.make_archive("/kaggle/working/outputs", "zip", out)
!du -sh /kaggle/working/outputs.zip